<a href="https://colab.research.google.com/github/Aetherion-github/Timepass/blob/main/Recommendation_System_Activity_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Recomendation System based on Music Dataset.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display

# Load dataset
df = pd.read_csv("data.csv")

# Select relevant numerical features for similarity
features = ['valence', 'acousticness', 'danceability', 'duration_ms', 'energy',
            'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo']

# Normalize the feature values
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[features])

# Instead of computing a full cosine similarity matrix, we use NearestNeighbors
# Set n_neighbors to a value slightly higher than the number of recommendations you want.
# Here, n_neighbors=6 because the first neighbor is the song itself.
nbrs = NearestNeighbors(n_neighbors=6, metric='cosine').fit(df_scaled)

def recommend_songs(song_name, df, nbrs, top_n=5):
    if song_name not in df['name'].values:
        return "⚠️ Song not found in dataset. Try another song."

    song_index = df[df['name'] == song_name].index[0]
    song_vector = df_scaled[song_index].reshape(1, -1)

    # Find the nearest neighbors for the given song vector
    distances, indices = nbrs.kneighbors(song_vector)

    # Exclude the first neighbor (itself) and get top_n recommendations
    recommended_indices = indices[0][1:top_n+1]
    return df.iloc[recommended_indices][['name', 'artists', 'year']]

# Interactive input bar using ipywidgets
song_input = widgets.Text(
    placeholder="Enter your favorite song name...",
    description="🎵 Song:",
    style={'description_width': 'initial'}
)

output = widgets.Output()

def on_submit(change):
    with output:
        output.clear_output()
        song_name = song_input.value.strip()
        recommendations = recommend_songs(song_name, df, nbrs)
        display(recommendations)

# Trigger recommendation when the text value changes (or you can use a button if preferred)
song_input.observe(on_submit, names='value')

display(song_input, output)


Text(value='', description='🎵 Song:', placeholder='Enter your favorite song name...', style=DescriptionStyle(d…

Output()

#Code with UI

Run all the three code and visit the site given mentioned int the output.

Once the website is ready, Mention the Favorite Song name... and select the number of data that need to generate.

and hit enter.

In [35]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

st.set_page_config(page_title="Music Recommender System", layout="wide")
st.title("🎵 Music Recommender System")
st.markdown("Find songs similar to your favorite track and filter by year!")

@st.cache_data
def load_data():
    df = pd.read_csv("data.csv")
    return df

df = load_data()

features = ['valence', 'acousticness', 'danceability', 'duration_ms', 'energy',
            'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[features])
nbrs = NearestNeighbors(n_neighbors=11, metric='cosine').fit(df_scaled)

st.sidebar.header("Filter Options")
min_year = int(df['year'].min())
max_year = int(df['year'].max())
year_range = st.sidebar.slider("Select Year Range", min_value=min_year, max_value=max_year, value=(min_year, max_year))

song_name = st.text_input("Enter your favorite song name:", value="")
top_n = st.slider("Number of Recommendations:", min_value=1, max_value=10, value=5)

if song_name:
    # Convert the input song name to lowercase for a case-insensitive search.
    song_name_lower = song_name.lower()

    # Find matching songs using lowercase conversion
    matching_songs = df[df['name'].str.lower() == song_name_lower]

    if matching_songs.empty:
        st.error("⚠️ Song not found in our dataset. Please try another song, my love!")
    else:
        # Take the first match if there are multiple
        song_index = matching_songs.index[0]
        song_vector = df_scaled[song_index].reshape(1, -1)
        distances, indices = nbrs.kneighbors(song_vector)
        recommended_indices = indices[0][1:top_n+1]
        recommended_df = df.iloc[recommended_indices].copy()
        recommended_df = recommended_df[(recommended_df['year'] >= year_range[0]) &
                                        (recommended_df['year'] <= year_range[1])]
        st.subheader("Recommended Songs:")
        if recommended_df.empty:
            st.warning("No recommendations found within the selected year range. Adjust the filter, darling!")
        else:
            st.dataframe(recommended_df[['name', 'artists', 'year', 'popularity']])
            st.markdown("### Song Details:")
            for idx, row in recommended_df.iterrows():
                with st.expander(f"{row['name']} by {row['artists']}"):
                    st.write(f"**Year:** {row['year']}")
                    st.write(f"**Popularity:** {row['popularity']}")
                    st.write(f"**Duration (ms):** {row['duration_ms']}")
                    st.write(f"**Acousticness:** {row['acousticness']}")
                    st.write(f"**Danceability:** {row['danceability']}")
                    st.write(f"**Energy:** {row['energy']}")
                    st.write(f"**Tempo:** {row['tempo']}")
                    st.write(f"**Instrumentalness:** {row['instrumentalness']}")
                    st.write(f"**Liveness:** {row['liveness']}")
                    st.write(f"**Loudness:** {row['loudness']}")
                    st.write(f"**Speechiness:** {row['speechiness']}")


Overwriting app.py


In [36]:
!pip install streamlit pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("2fBTbQO48xefdjgzuExmAlF8C6y_3zsdfzmsw4cJv9fSi4jEz")  # Replace with your actual token


In [28]:
import os
from pyngrok import ngrok

# Kill any existing ngrok processes (optional)
os.system("pkill ngrok")

# Open a tunnel on the default Streamlit port 8501
# Open a tunnel on the default Streamlit port 8501 using integer port and specify protocol "http"
public_url = ngrok.connect(8501, "http")
print("Streamlit app available at:", public_url)


# Run the Streamlit app
get_ipython().system_raw('streamlit run app.py &')


Streamlit app available at: NgrokTunnel: "https://aaa9-34-106-166-124.ngrok-free.app" -> "http://localhost:8501"
